# Range cover: main notebook

Run from the top to the bottom in Colab with a **T4 GPU** (Runtime > Change runtime type).
All real runs happen here. The Mac only runs `--smoke` and `pytest` as fast checks.

Each step is one cell. Each script resumes, so after a disconnect, run the setup cell again and then the cell that stopped.

## Setup
From `instructions.md`. Run again after a disconnect or restart.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['DATA_DIR'] = '/content/drive/MyDrive/cv_capstone_project/data'
os.environ['UV_PROJECT_ENVIRONMENT'] = '/content/.venv'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

%cd /content/drive/MyDrive/cv_capstone_project
!pip install -q uv
!uv sync --frozen

Mounted at /content/drive
/content/drive/MyDrive/cv_capstone_project
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: /content/.venv
Prepared 98 packages in 1m 07s
Installed 98 packages in 874ms
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + anyio==4.16.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.9.0
 + cuda-toolkit==13.2.2
 + cv-capstone-project==0.1.0 (from file:///content/drive/MyDrive/cv_capstone_project)
 + earthengine-api==1.7.50
 + filelock==4.2.10
 + fsspec==2026.9.0
 + google-api-core==2.43.2
 + google-api-python-client==2.201.0
 + google-auth==2.62.0
 + google-auth-httplib2==0.4.4
 + google-cloud-core==2.8.0
 + google-cloud-storage==3.17.1
 + google-crc32c==1.9.0
 + google-resumable-media==2.11.0
 + googleapis-common-protos==1.76.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.2
 + httplib2==0.3

## Check the code version
Drive must have synced the latest code from the Mac. The times must match the files on the Mac.

In [2]:
!ls -l --time-style=+%F_%T src/rangecover/*.py tests/*.py

-rw------- 1 root root 8590 2026-11-16_16:08:28 src/rangecover/dinov2.py
-rw------- 1 root root 4285 2026-11-16_14:40:42 src/rangecover/evaluate.py
-rw------- 1 root root   71 2026-11-16_14:39:42 src/rangecover/__init__.py
-rw------- 1 root root 5871 2026-11-16_14:43:26 src/rangecover/splits.py
-rw------- 1 root root 2227 2026-11-16_14:41:28 tests/test_evaluate.py
-rw------- 1 root root 1898 2026-11-16_14:41:28 tests/test_splits.py


## Step 2a: DINOv2 features (GPU only when visits are left)
First, the script looks on Drive for visits that already have features. It copies and runs only the visits that are left (16 photos copied at the same time, with progress). When all 358 visits are done, this cell copies nothing, loads no model and takes seconds.
Output: `data/dinov2/photos/*.npy` and `data/dinov2_visit_mean.npz` (358 × 768). On a CPU-only runtime, the `nvidia-smi` line prints an error; the cell still runs.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!uv run --frozen python -m rangecover.dinov2 --local /content/photos

name, memory.total [MiB]
Tesla T4, 15360 MiB
358/358 visits already have features; 0 to do
50/358 visits
100/358 visits
150/358 visits
200/358 visits
250/358 visits
300/358 visits
350/358 visits
358/358 visits
Saved (358, 768) to /content/drive/MyDrive/cv_capstone_project/data/dinov2_visit_mean.npz (device: cpu)


In [4]:
import numpy as np
f = np.load(os.environ['DATA_DIR'] + '/dinov2_visit_mean.npz')
print(f['features'].shape, f['features'].dtype, 'finite:', np.isfinite(f['features']).all())

(358, 768) float32 finite: True


## Step 2b: tests
Invariant 1: no site in both train and test. Invariant 2: predictions add to 100.

In [5]:
!uv run --frozen pytest -q

............                                                             [100%]
12 passed in 11.18s


## Step 2b: the 15 test runs and the dummy model
The dummy model always predicts the mean of the training labels. Every real method must have a lower error.

In [6]:
!uv run --frozen python -m rangecover.splits
!uv run --frozen python -m rangecover.evaluate

358 visits, 336 sites, 30 blocks
run  0  seed 0  group 0  test  73 visits /  69 sites  train 285
run  1  seed 0  group 1  test  70 visits /  70 sites  train 288
run  2  seed 0  group 2  test  69 visits /  66 sites  train 289
run  3  seed 0  group 3  test  76 visits /  61 sites  train 282
run  4  seed 0  group 4  test  70 visits /  70 sites  train 288
run  5  seed 1  group 0  test  78 visits /  73 sites  train 280
run  6  seed 1  group 1  test  73 visits /  59 sites  train 285
run  7  seed 1  group 2  test  67 visits /  67 sites  train 291
run  8  seed 1  group 3  test  71 visits /  71 sites  train 287
run  9  seed 1  group 4  test  69 visits /  66 sites  train 289
run 10  seed 2  group 0  test  68 visits /  67 sites  train 290
run 11  seed 2  group 1  test  70 visits /  66 sites  train 288
run 12  seed 2  group 2  test  76 visits /  73 sites  train 282
run 13  seed 2  group 3  test  75 visits /  75 sites  train 283
run 14  seed 2  group 4  test  69 visits /  55 sites  train 289
     ma

## Step 3: features (check)
Builds the training and test feature table of each of the 15 runs and prints their sizes. Writes nothing.
Columns: 8 satellite (6 band means, NDVI, NDTI), 3 weather (rain over the last 3, 6 and 12 months) and 20 photo columns. The photo PCA is fit on the training visits of each run only. Step 4 builds the same tables again inside its loop.

In [ ]:
!uv run --frozen python -m rangecover.features

## Step 4: the four methods on the 15 test runs
#1 RF satellite, #2 RF satellite + weather, #3 RF satellite + weather + photos, #4 linear (ridge) on all inputs. One loop; the methods differ only in model kind and input columns. CPU only, no GPU needed. It prints one line per run with the dry MAE of each method (about 1 s per run on a 10-core Mac; Colab has fewer CPUs, so the printed times show the real rate).
Output: `data/predictions.csv` (not committed) and `results/run_errors.csv` (committed). It always trains again, so a rerun checks that the numbers can be made again.

In [ ]:
!uv run --frozen python -m rangecover.models

## Step 5: results table, chart and verdict
The verdict compares #3 with #2 only, with the pass mark fixed on 15 Nov. Output in `results/`: `results_table.csv`, `verdict.json`, `dry_mae_by_run.png`, `results.md`.

In [ ]:
!uv run --frozen python -m rangecover.results

from IPython.display import Image, Markdown, display
page = open('results/results.md').read().replace('![Dry MAE per test run](dry_mae_by_run.png)', '')
display(Markdown(page))
display(Image('results/dry_mae_by_run.png', width=900))

## End
Make this the last cell, so all writes reach Drive.

In [7]:
drive.flush_and_unmount()